In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [8]:
def quantum_random_bits(n):
    """Generate n random bits by measuring |+> = (|0>+|1>)/sqrt(2)"""
    bits = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)

        simulator = AerSimulator()
        compiled = transpile(qc, simulator)
        job = simulator.run(compiled, shots=1)
        result = job.result()
        counts = result.get_counts()
        bit = int(list(counts.keys())[0])
        bits.append(bit)
    return bits
# SIFTING

def sift_key(bits, alice_bases, bob_bases):
    """Keep bits where Alice and Bob used the same basis."""
    sifted = []
    matching_indices = []
    for i, (ab, bb) in enumerate(zip(alice_bases, bob_bases)):
        if ab == bb:
            sifted.append(bits[i])
            matching_indices.append(i)
    return sifted, matching_indices

# ERROR CHECKING

def check_error_rate(alice_key, bob_key, sample_size=None):
    """
    Compare a sample of Alice's and Bob's sifted keys.
    Returns error rate. If > threshold → attack suspected.
    """
    if sample_size is None:
        sample_size = len(alice_key) // 4

    errors = 0
    for i in range(sample_size):
        if alice_key[i] != bob_key[i]:
            errors += 1

    return errors / sample_size if sample_size > 0 else 0

# ALICE: Encode qubits
def alice_encode(bits, bases):
    """
    Alice encodes each bit into a qubit.
    bases: 0 = Z-basis (|0>, |1>), 1 = X-basis (|+>, |->)
    bits:  0 or 1
    """
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits

# BOB: Measure qubits

def bob_measure(circuits, bases):
    """
    Bob measures each qubit in his randomly chosen basis.
    bases: 0 = Z-basis, 1 = X-basis
    """
    results = []
    simulator = AerSimulator()

    for qc, basis in zip(circuits, bases):
        measure_qc = qc.copy()
        if basis == 1:
            measure_qc.h(0)
        measure_qc.measure(0, 0)

        compiled = transpile(measure_qc, simulator)
        job = simulator.run(compiled, shots=1)
        result = job.result()
        counts = result.get_counts()
        bit = int(list(counts.keys())[0])
        results.append(bit)
    return results

def attacker_intercept(circuits):
    """
    Eve intercepts each qubit, measures in a random basis (her best guess),
    then re-encodes and resends.
    She has no idea what basis Alice used — so she'll be wrong ~50% of the time.
    When wrong, she disturbs the qubit state → causes errors Bob can detect.
    """
    print("[EVE]   Intercepting and resending qubits...")
    eve_bases    = quantum_random_bits(len(circuits))   # Eve's random guesses
    eve_results  = []
    new_circuits = []

    simulator = AerSimulator()

    for qc, basis in zip(circuits, eve_bases):
        # Eve measures in her chosen basis
        intercept_qc = qc.copy()
        if basis == 1:
            intercept_qc.h(0)
        intercept_qc.measure(0, 0)

        compiled = transpile(intercept_qc, simulator)
        job = simulator.run(compiled, shots=1)
        counts = job.result().get_counts()
        measured_bit = int(list(counts.keys())[0])
        eve_results.append(measured_bit)

        # Eve re-encodes what she measured and sends it on to Bob
        new_qc = QuantumCircuit(1, 1)
        if measured_bit == 1:
            new_qc.x(0)
        if basis == 1:
            new_qc.h(0)
        new_circuits.append(new_qc)

    return new_circuits, eve_results, eve_bases


# MAIN PROTOCOL — WITH ATTACKER (EVE)

NUM_QUBITS = 100
THRESHOLD  = 0.1

print("=" * 55)
print("    BB84 Protocol — With Attacker (Eve)")
print("=" * 55)

# ALICE
print("\n[ALICE] Generating random bits and bases...")
alice_bits  = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)
print("[ALICE] Encoding qubits...")
encoded_qubits = alice_encode(alice_bits, alice_bases)

# EVE intercepts!
intercepted_qubits, eve_results, eve_bases = attacker_intercept(encoded_qubits)

# BOB measures EVE's resent qubits
print("[BOB]   Generating random bases and measuring...")
bob_bases   = quantum_random_bits(NUM_QUBITS)
bob_results = bob_measure(intercepted_qubits, bob_bases)

# SIFTING
print("\n[PUBLIC] Alice and Bob compare bases...")
alice_sifted, _ = sift_key(alice_bits,  alice_bases, bob_bases)
bob_sifted,   _ = sift_key(bob_results, alice_bases, bob_bases)

print(f"  Qubits sent:    {NUM_QUBITS}")
print(f"  Matching bases: {len(alice_sifted)}")

# ERROR CHECKING
error_rate = check_error_rate(alice_sifted, bob_sifted)
print(f"\n[CHECK] Error rate on sample: {error_rate*100:.1f}%")

if error_rate > THRESHOLD:
    print("  ATTACK DETECTED! Error rate is suspiciously high. Aborting key exchange.")
    print(f"       (Expected ~25% errors with intercept-and-resend, got {error_rate*100:.1f}%)")
else:
    print("  No attack detected (Eve got lucky this run — try again!)")

    BB84 Protocol — With Attacker (Eve)

[ALICE] Generating random bits and bases...
[ALICE] Encoding qubits...
[EVE]   Intercepting and resending qubits...
[BOB]   Generating random bases and measuring...

[PUBLIC] Alice and Bob compare bases...
  Qubits sent:    100
  Matching bases: 52

[CHECK] Error rate on sample: 15.4%
  ATTACK DETECTED! Error rate is suspiciously high. Aborting key exchange.
       (Expected ~25% errors with intercept-and-resend, got 15.4%)
